# dlt Workshop Homework

This notebook documents the verified Pydantic AI + Logfire + dlt workshop run and answers the three homework questions from the exported DuckDB trace data.

## 1. Setup

We work from the already exported Logfire trace and the loaded DuckDB dataset. No additional LLM calls are required to reproduce the final analysis in this notebook.

In [1]:

from pathlib import Path
import duckdb
import pandas as pd

TRACE_ID = '019f8080cd4c27a64caf5b4a812a5671'
QUESTION = 'How do I run Ollama locally?'
ANSWER_PREVIEW = "Based on the FAQ, here's how to run Ollama locally: ..."
DB_PATH = Path('workshop_logfire.duckdb')
assert DB_PATH.exists(), DB_PATH


## 2. Agent Configuration

The workshop agent uses the FAQ index, the official teaching-assistant instruction, and a Qwen model through an OpenAI-compatible provider.

In [2]:

print('Question:', QUESTION)
print('Trace ID:', TRACE_ID)
print('Answer preview:', ANSWER_PREVIEW)


Question: How do I run Ollama locally?
Trace ID: 019f8080cd4c27a64caf5b4a812a5671
Answer preview: Based on the FAQ, here's how to run Ollama locally: ...


## 3. Logfire Instrumentation

The measured run was exported to Logfire successfully after the write-token fix. The resulting Logfire records were then pulled into DuckDB with dlt.

## 4. Agent Run for `How do I run Ollama locally?`

We inspect the normalized `records` table for the final measured trace.

In [3]:

con = duckdb.connect(DB_PATH.as_posix(), read_only=True)
trace_df = con.execute(
    f"""
    SELECT trace_id, span_name, parent_span_id, duration,
           attributes__gen_ai_usage_input_tokens,
           attributes__gen_ai_usage_output_tokens,
           attributes__gen_ai_aggregated_usage_input_tokens,
           attributes__gen_ai_aggregated_usage_output_tokens,
           attributes__gen_ai_tool_name
    FROM agent_traces.records
    WHERE trace_id = '{TRACE_ID}'
    ORDER BY start_timestamp
    """
).fetchdf()
trace_df


,trace_id,span_name,parent_span_id,duration,attributes__gen_ai_usage_input_tokens,attributes__gen_ai_usage_output_tokens,attributes__gen_ai_aggregated_usage_input_tokens,attributes__gen_ai_aggregated_usage_output_tokens,attributes__gen_ai_tool_name
0,019f8080cd4c27a64caf5b4a812a5671,invoke_agent agent,NaN,18.053139,<NA>,<NA>,2197,395,NaN
1,019f8080cd4c27a64caf5b4a812a5671,chat qwen-plus,7b995b7acf36ec12,9.644610,326,73,<NA>,<NA>,NaN
2,019f8080cd4c27a64caf5b4a812a5671,execute_tool search,7b995b7acf36ec12,0.053392,<NA>,<NA>,<NA>,<NA>,search
3,019f8080cd4c27a64caf5b4a812a5671,chat qwen-plus,7b995b7acf36ec12,8.305219,1871,322,<NA>,<NA>,NaN


## 5. dlt Extraction from Logfire

The dlt pipeline normalized the exported Logfire records into the `agent_traces` schema inside DuckDB.

In [4]:

tables_df = con.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'agent_traces' ORDER BY table_name"
).fetchdf()
tables_df.head(10)


,table_name
0,_dlt_loads
1,_dlt_pipeline_state
2,_dlt_version
3,records
4,records__attributes__gen_ai_input_messages
5,records__attributes__gen_ai_input_messages__parts
6,records__attributes__gen_ai_input_messages__pa...
7,records__attributes__gen_ai_output_messages
8,records__attributes__gen_ai_output_messages__p...
9,records__attributes__gen_ai_response_finish_re...


## 6. DuckDB Table Inspection

For Q2 we count the normalized tables created in the `agent_traces` schema.

In [5]:

q2_table_count = len(tables_df)
print('Table count:', q2_table_count)
print('All table names:')
for name in tables_df['table_name']:
    print('-', name)


Table count: 23
All table names:
- _dlt_loads
- _dlt_pipeline_state
- _dlt_version
- records
- records__attributes__gen_ai_input_messages
- records__attributes__gen_ai_input_messages__parts
- records__attributes__gen_ai_input_messages__parts__result
- records__attributes__gen_ai_output_messages
- records__attributes__gen_ai_output_messages__parts
- records__attributes__gen_ai_response_finish_reasons
- records__attributes__gen_ai_system_instructions
- records__attributes__gen_ai_tool_call_result
- records__attributes__gen_ai_tool_definitions
- records__attributes__gen_ai_tool_definitions__parameters__required
- records__attributes__logfire_metrics__gen_ai_client_token_usage__details
- records__attributes__logfire_scrubbed
- records__attributes__logfire_scrubbed__path
- records__attributes__model_request_parameters__function_tools
- records__attributes__model_request_parameters__function_tools__parameters_json_schema__required
- records__attributes__model_request_parameters__instruction_

## 7. Span-Count Analysis

For Q1 we count the rows in the `records` table for the single measured trace.

In [6]:

q1_span_count = len(trace_df)
span_names = trace_df['span_name'].tolist()
print('Span count:', q1_span_count)
print('Span names:', span_names)


Span count: 4
Span names: ['invoke_agent agent', 'chat qwen-plus', 'execute_tool search', 'chat qwen-plus']


## 8. Token-Usage Analysis

For Q3 we sum `gen_ai.usage.input_tokens` across the LLM spans in the measured trace.

In [7]:

llm_df = trace_df[trace_df['span_name'] == 'chat qwen-plus'].copy()
llm_input_tokens = llm_df['attributes__gen_ai_usage_input_tokens'].astype(int).tolist()
llm_output_tokens = llm_df['attributes__gen_ai_usage_output_tokens'].astype(int).tolist()
q3_total_input_tokens = sum(llm_input_tokens)
print('LLM input tokens per span:', llm_input_tokens)
print('LLM output tokens per span:', llm_output_tokens)
print('Total input tokens:', q3_total_input_tokens)


LLM input tokens per span: [326, 1871]
LLM output tokens per span: [73, 322]
Total input tokens: 2197


## 9. Final Answers

The final outputs are mapped to the workshop multiple-choice options.

In [8]:

final_answers = {
    'Q1_exact_span_count': q1_span_count,
    'Q1_option': 5,
    'Q2_exact_table_count': q2_table_count,
    'Q2_option': 24,
    'Q3_input_tokens_per_llm_span': llm_input_tokens,
    'Q3_total_input_tokens': q3_total_input_tokens,
    'Q3_option': '1500 - 5000',
}
final_answers


{'Q1_exact_span_count': 4,
 'Q1_option': 5,
 'Q2_exact_table_count': 23,
 'Q2_option': 24,
 'Q3_input_tokens_per_llm_span': [326, 1871],
 'Q3_total_input_tokens': 2197,
 'Q3_option': '1500 - 5000'}